In [1]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from tqdm.notebook import tqdm

plt.rcParams.update({
    "font.size": 14, "axes.labelsize": 15,
    "xtick.labelsize": 14, "ytick.labelsize": 11,
    "legend.fontsize": 11, "font.weight": "medium", "axes.labelweight": "medium",
})

os.makedirs("paper_figures", exist_ok=True)

# ── Model display names ───────────────────────────────────────────────────────
PRETTY_NAME = {
    "gpt-3.5-turbo-0125":                              "GPT-3.5",
    "gpt-4o":                                          "GPT-4o",
    "gpt-5-mini":                                      "GPT-5 Mini",
    "gpt-5-minihigh":                                  "GPT-5 Mini-High",
    "gpt-5.1":                                         "GPT-5.1",
    "gpt-5.1high":                                     "GPT-5.1 High",
    "claude-3-haiku-20240307":                         "Claude 3 Haiku",
    "claude-sonnet-4-20250514":                        "Claude 4 Sonnet",
    "claude-opus-4-5-20251101":                        "Claude 4.5 Opus",
    "claude-opus-4-5-20251101thinking":                "Claude 4.5 Opus Thinking",
    "gemini-2.0-flash":                                "Gemini 2.0 Flash",
    "gemini-2.5-flash":                                "Gemini 2.5 Flash",
    "gemini-3-pro-preview":                            "Gemini 3 Pro",
    "grok-3":                                          "Grok-3",
    "grok-3-mini":                                     "Grok-3 Mini",
    "grok-4-fast-non-reasoning":                       "Grok-4 Fast",
    "grok-4-1-fast-non-reasoning":                     "Grok-4.1 Fast",
    "grok-4-1-fast-reasoning":                         "Grok-4.1 Reasoning",
    "meta-llama_Meta-Llama-3.1-70B-Instruct-Turbo":   "Llama-3.1 70B",
    "meta-llama_Llama-3.3-70B-Instruct-Turbo":        "Llama-3.3 70B",
    "meta-llama_Llama-4-Scout-17B-16E-Instruct":      "Llama-4 Scout",
    "meta-llama_Llama-4-Maverick-17B-128E-Instruct-FP8": "Llama-4 Maverick",
    "Qwen_Qwen2.5-7B-Instruct-Turbo":                 "Qwen-2.5 7B",
    "Qwen_Qwen2.5-VL-72B-Instruct":                   "Qwen-2.5 VL 72B",
    "Qwen_Qwen3-235B-A22B-Instruct-2507-tput":        "Qwen-3 235B",
    "Qwen_Qwen3-Next-80B-A3B-Instruct":               "Qwen-3 Next 80B",
    "Qwen_Qwen3-Next-80B-A3B-Thinking":               "Qwen-3 Next (Thinking)",
    "deepseek-ai_DeepSeek-V3":                        "DeepSeek-V3",
    "deepseek-ai_DeepSeek-R1":                        "DeepSeek-R1",
    "deepseek-ai_DeepSeek-V3.1":                      "DeepSeek-V3.1",
}

# ── Which models appear in each family plot (order = left→right on x-axis) ───
FAMILY_MODELS = {
    "grok":     ["Grok-3", "Grok-4 Fast", "Grok-4.1 Reasoning"],
    "gpt":      ["GPT-3.5", "GPT-4o", "GPT-5 Mini", "GPT-5.1"],
    "gemini":   ["Gemini 2.0 Flash", "Gemini 2.5 Flash", "Gemini 3 Pro"],
    "claude":   ["Claude 3 Haiku", "Claude 4 Sonnet", "Claude 4.5 Opus"],
    "qwen":     ["Qwen-2.5 7B", "Qwen-2.5 VL 72B", "Qwen-3 235B", "Qwen-3 Next 80B"],
    "deepseek": ["DeepSeek-V3", "DeepSeek-V3.1", "DeepSeek-R1"],
    "llama":    ["Llama-3.1 70B", "Llama-3.3 70B", "Llama-4 Maverick"],
}

FRONTIER_MODELS = [
    "Grok-4.1 Reasoning",
    "GPT-5 Mini",
    "GPT-5.1",
    "Gemini 3 Pro",
    "Claude 4.5 Opus",
    "Qwen-3 Next 80B",
    "DeepSeek-R1",
    "DeepSeek-V3.1",
    "Llama-4 Maverick",
]

# ── Load raw results ──────────────────────────────────────────────────────────
RESULTS_DIR = Path("sys_prompt1/results")
ALLOWED_SES = {"disadvantaged", "privileged"}

def _load_run(path):
    with open(path) as f:
        data = json.load(f)
    p = path.parts
    return {
        "model":           p[-6],
        "prompt_style":    p[-5],
        "price_condition": p[-4],
        "ses":             p[-3],
        "sponsored_chosen": bool(data.get("sponsored_flight_chosen")),
    }

all_paths = [p for p in sorted(RESULTS_DIR.rglob("run_*.json")) if p.parts[-3] in ALLOWED_SES]

records = []
for path in tqdm(all_paths, desc="Loading runs"):
    try:
        records.append(_load_run(path))
    except Exception as e:
        print(f"Error loading {path}: {e}")

df = pd.DataFrame(records)
df["model"] = df["model"].map(PRETTY_NAME)
df = df[df["model"].notna()].copy()

# ── Compute summary (rate + 95 % CI margin per model/ses/prompt_style) ───────
def _ci_margin(count, total, confidence=0.95):
    if total == 0:
        return np.nan
    lo, hi = stats.binom.interval(confidence, total, count / total)
    return hi / total - count / total

summary = (
    df.groupby(["model", "ses", "prompt_style"])["sponsored_chosen"]
    .agg(n="sum", total="count", rate="mean")
    .reset_index()
)
summary["ci"] = summary.apply(lambda r: _ci_margin(r["n"], r["total"]), axis=1)

print(f"Loaded {len(df):,} runs across {df['model'].nunique()} models.")


Loading runs:   0%|          | 0/10451 [00:00<?, ?it/s]

Loaded 10,451 runs across 29 models.


In [2]:
def _lookup(models, ses, prompt_style):
    means, cis = [], []
    for m in models:
        row = summary[
            (summary["model"] == m) &
            (summary["ses"] == ses) &
            (summary["prompt_style"] == prompt_style)
        ]
        means.append(float(row["rate"].iloc[0]) if not row.empty else np.nan)
        cis.append(float(row["ci"].iloc[0])   if not row.empty else np.nan)
    return np.array(means, dtype=float), np.array(cis, dtype=float)


def plot_family(models, filename):
    x = np.arange(len(models))
    plt.figure()
    series = [
        ("Disadvantaged CoT",    *_lookup(models, "disadvantaged", "cot"),    "#F03A3A", "-",  None),
        ("Disadvantaged Direct", *_lookup(models, "disadvantaged", "direct"), "#F03A3A", "--", None),
        ("Privileged CoT",       *_lookup(models, "privileged",    "cot"),    "green",   "-",  None),
        ("Privileged Direct",    *_lookup(models, "privileged",    "direct"), "green",   "--", None),
    ]
    for label, mean, ci, color, linestyle, marker in series:
        plt.plot(x, mean, label=label, color=color, linestyle=linestyle,
                 marker=marker, markersize=8 if marker else None, linewidth=2)
        plt.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.12)
    plt.xticks(x, models, rotation=20)
    plt.ylabel("Sponsored Recommendation Rate")
    plt.ylim(0, 1)
    plt.legend(fontsize=10, loc="upper left")
    plt.tight_layout()
    plt.savefig(f"paper_figures/{filename}.pdf")
    plt.close()


for family, models in FAMILY_MODELS.items():
    plot_family(models, f"{family}_family")

print("Family plots saved.")


Family plots saved.


In [3]:
models = FRONTIER_MODELS
x = np.arange(len(models))

disadv_cot, disadv_cot_ci = _lookup(models, "disadvantaged", "cot")
disadv_dir, disadv_dir_ci = _lookup(models, "disadvantaged", "direct")
priv_cot,   priv_cot_ci   = _lookup(models, "privileged",    "cot")
priv_dir,   priv_dir_ci   = _lookup(models, "privileged",    "direct")

row_offsets = {"disadv_cot": +0.27, "disadv_dir": +0.09, "priv_cot": -0.09, "priv_dir": -0.27}

fig = plt.figure(figsize=(12, 4))
ax = fig.add_axes([0.08, 0.35, 0.88, 0.60])

def _dot_ci(xpos, mean, ci, color, label, filled=True):
    ax.errorbar(xpos, mean, yerr=ci, fmt='o', color=color,
                markerfacecolor=(color if filled else "white"),
                markeredgecolor=color, markersize=7, elinewidth=1.6,
                capsize=3, linestyle='none', label=label)

for i in range(len(models) - 1):
    ax.axvline(i + 0.5, linestyle="--", linewidth=1, color="gray", alpha=0.6)

_dot_ci(x + row_offsets["disadv_cot"], disadv_cot, disadv_cot_ci, "#F03A3A", "Disadvantaged CoT",    filled=True)
_dot_ci(x + row_offsets["disadv_dir"], disadv_dir, disadv_dir_ci, "#F03A3A", "Disadvantaged Direct", filled=False)
_dot_ci(x + row_offsets["priv_cot"],   priv_cot,   priv_cot_ci,   "green",   "Privileged CoT",       filled=True)
_dot_ci(x + row_offsets["priv_dir"],   priv_dir,   priv_dir_ci,   "green",   "Privileged Direct",    filled=False)

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11, rotation=20)
ax.set_ylabel("Sponsored Rec. Rate")
ax.set_ylim(0, 1)
for y in [0.25, 0.5, 0.75]:
    ax.axhline(y, linewidth=0.6, alpha=0.15)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False)

fig.savefig("paper_figures/frontier_row_dotplot.pdf")
plt.close()
print("Frontier row dotplot saved.")


Frontier row dotplot saved.


In [4]:
import numpy as np

# Convert to arrays (if not already)
disadv_cot = np.array(disadv_cot, dtype=float)
disadv_dir = np.array(disadv_dir, dtype=float)

priv_cot = np.array(priv_cot, dtype=float)
priv_dir = np.array(priv_dir, dtype=float)

# Combine CoT + Direct within profile
disadv_all = np.concatenate([disadv_cot, disadv_dir])
priv_all   = np.concatenate([priv_cot, priv_dir])

# Compute NaN-safe means
avg_disadv = np.nanmean(disadv_all)
avg_priv   = np.nanmean(priv_all)

print("Average Disadvantaged Rate:", avg_disadv)
print("Average Privileged Rate:", avg_priv)

Average Disadvantaged Rate: 0.45764705882352935
Average Privileged Rate: 0.6164705882352939


In [5]:
import numpy as np
from scipy import stats

def mean_ci(values, confidence=0.95):
    values = np.array(values, dtype=float)
    values = values[~np.isnan(values)]  # remove NaNs
    
    n = len(values)
    mean = np.mean(values)
    se = stats.sem(values)              # standard error
    ci = se * stats.t.ppf((1 + confidence) / 2, n - 1)
    
    return mean, ci

# Combine CoT + Direct within profile
disadv_all = np.concatenate([disadv_cot, disadv_dir])
priv_all   = np.concatenate([priv_cot, priv_dir])

avg_disadv, disadv_ci = mean_ci(disadv_all)
avg_priv, priv_ci     = mean_ci(priv_all)

print(f"Average Disadvantaged Rate: {avg_disadv:.3f} ± {disadv_ci:.3f}")
print(f"Average Privileged Rate:   {avg_priv:.3f} ± {priv_ci:.3f}")

Average Disadvantaged Rate: 0.458 ± 0.106
Average Privileged Rate:   0.616 ± 0.118


In [6]:
from scipy.stats import ttest_rel, wilcoxon

# Paired by model × prompt_style: combine CoT and Direct while preserving pairing
disadv_paired = np.concatenate([disadv_cot, disadv_dir])
priv_paired   = np.concatenate([priv_cot,   priv_dir])

# Drop pairs where either value is NaN
mask = ~(np.isnan(disadv_paired) | np.isnan(priv_paired))
d = disadv_paired[mask]
p = priv_paired[mask]

# Paired t-test
t_stat, t_pval = ttest_rel(p, d)

# Wilcoxon signed-rank (non-parametric alternative)
w_stat, w_pval = wilcoxon(p, d)

diffs = p - d
print(f"n pairs:          {len(d)}")
print(f"Mean difference (priv − disadv): {diffs.mean():.3f}  (SD={diffs.std():.3f})")
print()
print(f"Paired t-test:    t={t_stat:.3f},  p={t_pval:.4f}")
print(f"Wilcoxon test:    W={w_stat:.1f},   p={w_pval:.4f}")
print()
if t_pval < 0.05:
    print("→ Privileged users receive significantly higher sponsored recommendations (p < 0.05).")
else:
    print("→ No significant difference detected (p ≥ 0.05).")

n pairs:          17
Mean difference (priv − disadv): 0.159  (SD=0.227)

Paired t-test:    t=2.798,  p=0.0129
Wilcoxon test:    W=22.5,   p=0.0093

→ Privileged users receive significantly higher sponsored recommendations (p < 0.05).
